In [1]:
import pandas as pd
import os
import pickle

from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

In [2]:
BASE_PATH = "../Data"

isot_df = pd.read_csv(os.path.join(BASE_PATH, "ISOT", "isot_cleaned.csv"))
welfake_df = pd.read_csv(os.path.join(BASE_PATH, "WELFake", "welfake_cleaned.csv"))

# Fix missing values (IMPORTANT)
isot_df["content"] = isot_df["content"].fillna("").astype(str)
welfake_df["content"] = welfake_df["content"].fillna("").astype(str)

print("ISOT Shape:", isot_df.shape)
print("WELFake Shape:", welfake_df.shape)

ISOT Shape: (38827, 2)
WELFake Shape: (62749, 2)


In [3]:
tfidf_isot = pickle.load(open("../models/tfidf_isot.pkl", "rb"))
tfidf_welfake = pickle.load(open("../models/tfidf_welfake.pkl", "rb"))

In [4]:
# ISOT
X_isot = tfidf_isot.transform(isot_df["content"])
y_isot = isot_df["label"]

# WELFake
X_welfake = tfidf_welfake.transform(welfake_df["content"])
y_welfake = welfake_df["label"]

In [5]:
# ISOT
X_train_isot, X_test_isot, y_train_isot, y_test_isot = train_test_split(
    X_isot, y_isot, test_size=0.2, random_state=42
)

# WELFake
X_train_wel, X_test_wel, y_train_wel, y_test_wel = train_test_split(
    X_welfake, y_welfake, test_size=0.2, random_state=42
)

In [6]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    ),
    
    "Naive Bayes": MultinomialNB(),
    
    "SVM": LinearSVC(
        class_weight='balanced'
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42
    ),
    
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
}

In [7]:
isot_trained_models = {}

for name, model in models.items():
    print(f"Training {name} on ISOT...")
    
    model.fit(X_train_isot, y_train_isot)
    
    isot_trained_models[name] = model

print("✅ All ISOT models trained")

Training Logistic Regression on ISOT...
Training Naive Bayes on ISOT...
Training SVM on ISOT...
Training Random Forest on ISOT...
Training KNN on ISOT...
Training XGBoost on ISOT...
✅ All ISOT models trained


In [8]:
welfake_trained_models = {}

for name, model in models.items():
    print(f"Training {name} on WELFake...")
    
    model.fit(X_train_wel, y_train_wel)
    
    welfake_trained_models[name] = model

print("✅ All WELFake models trained")

Training Logistic Regression on WELFake...
Training Naive Bayes on WELFake...
Training SVM on WELFake...
Training Random Forest on WELFake...
Training KNN on WELFake...
Training XGBoost on WELFake...
✅ All WELFake models trained


In [9]:
os.makedirs("../models", exist_ok=True)

pickle.dump(isot_trained_models, open("../models/isot_models.pkl", "wb"))
pickle.dump(welfake_trained_models, open("../models/welfake_models.pkl", "wb"))

print("✅ Models saved successfully")

✅ Models saved successfully
